# Robust Covariance Estimation with Tyler's Estimator

This notebook demonstrates how Tyler's M‑estimator produces a covariance matrix that is robust to outliers, leading to more stable minimum‑variance portfolios, especially during market crises.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from data_loader import fetch_asset_data, compute_returns
from tyler_estimator import tyler_estimator
from portfolio_optimizer import min_variance_portfolio
import visualization as viz

## 1. Load Data

We'll use a set of US stocks and a benchmark (SPY). The data period includes the March 2020 COVID crash to highlight robustness.

In [ ]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'JNJ', 'XOM', 'WMT']
prices = fetch_asset_data(symbols, start_date='2019-01-01', end_date='2020-12-31')
returns = compute_returns(prices)
returns.head()

## 2. Estimate Covariance Matrices

Compute both sample covariance and Tyler's robust covariance.

In [ ]:
X = returns.values

# Sample covariance
cov_sample = np.cov(X, rowvar=False)

# Tyler estimator
cov_tyler = tyler_estimator(X, max_iter=50, tol=1e-5)

print("Sample covariance condition number:", np.linalg.cond(cov_sample))
print("Tyler covariance condition number:", np.linalg.cond(cov_tyler))

## 3. Minimum Variance Portfolios

Compute weights for each covariance estimate.

In [ ]:
weights_sample = min_variance_portfolio(cov_sample)
weights_tyler = min_variance_portfolio(cov_tyler)

viz.plot_portfolio_weights(weights_sample, weights_tyler, symbols)

## 4. Backtest Portfolios

Simulate the out‑of‑sample performance using a rolling window. For simplicity, we'll do a one‑time portfolio constructed on the entire period and then evaluate returns.

In practice, you would rebalance periodically. Here we just show the cumulative returns of the static portfolios.

In [ ]:
# Portfolio returns (daily)
port_sample = returns @ weights_sample
port_tyler = returns @ weights_tyler

# Benchmark: SPY
spy = fetch_asset_data(['SPY'], start_date='2019-01-01', end_date='2020-12-31')
spy_ret = compute_returns(spy)

viz.plot_cumulative_returns(port_sample, port_tyler, benchmark=spy_ret)

## 5. Performance Metrics

Compute Sharpe ratio, volatility, and maximum drawdown to compare robustness.

In [ ]:
def metrics(returns, ann_factor=252):
    vol = returns.std() * np.sqrt(ann_factor)
    sharpe = returns.mean() * ann_factor / vol
    cum = (1 + returns).cumprod()
    mdd = (cum / cum.cummax() - 1).min()
    return {'Volatility': vol, 'Sharpe': sharpe, 'Max Drawdown': mdd}

print("Sample Covariance Portfolio:")
print(metrics(port_sample))
print("\nTyler Portfolio:")
print(metrics(port_tyler))